# 01 · Coverage checks

**Direction:** Soil and Crop Resilience Planning. **User:** Soil Conservation Office.
**Decision:** where soil programs would cut drought losses most.

Before committing to Iowa corn and Iowa soybeans (analysed separately), this notebook answers two questions:

1. **Check 1 — coverage.** How many rows per state and crop actually have a reported yield, and how complete is county reporting in Iowa across the 26 years?
2. **Check 2 — drought exposure.** How many county-years sit at each `max_drought_severity` level, especially D3+, and are the Iowa D3+ cases spread across years or concentrated in one (e.g. 2012)?

Rules followed here: missing yields are never filled with zero, and no rows are dropped without saying why.
Run this notebook with the kernel's working directory set to `notebooks/` (that is what `jupyter nbconvert --execute notebooks/01_coverage_checks.ipynb` from the repo root does), so `sys.path.insert(0, "..")` finds `src/`.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from src.data_loader import load_master, load_nass_raw

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

df = load_master()
print(f"{len(df):,} rows × {len(df.columns)} columns")

17,056 rows × 21 columns


## Column names and codes

Real column names (verified against `load_master()` and `data/data_dictionary.md`) and the drought coding are kept as named variables so they are easy to change.

From the data dictionary, `max_drought_severity` is:

> Worst drought category observed in any week of the season, where more than 1% of the county was affected. 0 = no drought, 1 = D0 (abnormally dry), 2 = D1 (moderate), 3 = D2 (severe), 4 = D3 (extreme), 5 = D4 (exceptional). Note: this scale is offset by 1 from the USDM's own D0–D4 labels.

So **D2 = 3, D3 = 4, D4 = 5**. NaN means the season window was not fully covered by the USDM archive (only `WHEAT` year 2000, 150 rows).

In [2]:
# ---- Column names (master dataset) ----
COL_FIPS = "fips"                    # 5-char string county FIPS, keep as text
COL_YEAR = "year"                    # harvest year, int
COL_STATE = "state_alpha"            # CA / DE / IA / NE
COL_CROP = "crop"                    # CORN / SOYBEANS / WHEAT / SORGHUM
COL_YIELD = "yield_per_acre"         # bu/acre, NaN when not reported or suppressed
COL_YIELD_STATUS = "yield_status"    # reported / not_reported / suppressed
COL_SEV = "max_drought_severity"     # 0-5, see SEV_LABELS

# ---- Drought coding (0-5 scale, offset by +1 from USDM D0-D4) ----
SEV_LABELS = {0: "none", 1: "D0", 2: "D1", 3: "D2", 4: "D3", 5: "D4"}
SEV_D2 = 3   # severe
SEV_D3 = 4   # extreme
SEV_D4 = 5   # exceptional

# ---- Focus ----
FOCUS_STATE = "IA"
FOCUS_CROPS = ["CORN", "SOYBEANS"]
N_YEARS_EXPECTED = 26   # 2000-2025

# Known figures to sanity-check against (from the data dictionary / README)
KNOWN = {
    "rows_total": 17_056,
    "rows_with_yield": 11_639,
    "pct_with_yield": 68,
    "not_reported": 5_343,
    "suppressed": 74,
    "iowa_corn_reported_approx": 2_500,
    "delaware_corn_reported": 78,
}

In [3]:
# Guard rails: fips must be text, the columns must exist, and the severity codes must be what we assume.
for c in [COL_FIPS, COL_YEAR, COL_STATE, COL_CROP, COL_YIELD, COL_YIELD_STATUS, COL_SEV]:
    assert c in df.columns, f"missing column: {c}"
assert df[COL_FIPS].map(type).eq(str).all(), "fips is not text"
assert df[COL_FIPS].str.len().eq(5).all(), "fips is not 5 characters"

print(df.dtypes)
print()
print("states:", sorted(df[COL_STATE].unique()))
print("crops: ", sorted(df[COL_CROP].unique()))
print("years: ", df[COL_YEAR].min(), "-", df[COL_YEAR].max(), f"({df[COL_YEAR].nunique()} distinct)")
print()
print("max_drought_severity values (incl. NaN):")
print(df[COL_SEV].value_counts(dropna=False).sort_index())
print()
print("yield_status values:")
print(df[COL_YIELD_STATUS].value_counts(dropna=False))
print()
print("Irrigation-related columns in the master:", [c for c in df.columns if "irr" in c.lower()] or "none")

fips                     object
county_name              object
state_alpha              object
year                      int64
crop                     object
yield_per_acre          float64
yield_status             object
acres_planted           float64
acres_harvested         float64
yield_anomaly_pct       float64
precip_mm               float64
precip_anomaly_pct      float64
tavg_c                  float64
extreme_heat_days       float64
gdd                     float64
max_drought_severity    float64
weeks_in_d2_plus        float64
mean_dsci               float64
aws_100cm_mm            float64
droughty_pct            float64
nccpi_crop              float64
dtype: object

states: ['CA', 'DE', 'IA', 'NE']
crops:  ['CORN', 'SORGHUM', 'SOYBEANS', 'WHEAT']
years:  2000 - 2025 (26 distinct)

max_drought_severity values (incl. NaN):
max_drought_severity
0.0    2279
1.0    4134
2.0    3604
3.0    3517
4.0    2370
5.0    1002
NaN     150
Name: count, dtype: int64

yield_status values:
yi

In [4]:
# Confirm the coding empirically: severity >= SEV_D2 should coincide exactly with weeks_in_d2_plus > 0,
# and NaN severity should only occur for WHEAT year 2000.
chk = pd.crosstab(df[COL_SEV], df["weeks_in_d2_plus"] > 0)
chk.columns = ["weeks_in_d2_plus == 0", "weeks_in_d2_plus > 0"]
display(chk)

sev_nan = df[df[COL_SEV].isna()]
print("Rows with NaN severity, by crop and year:")
print(sev_nan.groupby([COL_CROP, COL_YEAR]).size())

assert (df.loc[df[COL_SEV] >= SEV_D2, "weeks_in_d2_plus"] > 0).all()
assert (df.loc[df[COL_SEV] < SEV_D2, "weeks_in_d2_plus"] == 0).all()
print("\nConfirmed: severity >= 3 <=> at least one week at D2 or worse. So 3 = D2, 4 = D3, 5 = D4.")

,weeks_in_d2_plus == 0,weeks_in_d2_plus > 0
max_drought_severity,,
0.0,2279,0
1.0,4134,0
2.0,3604,0
3.0,0,3517
4.0,0,2370
5.0,0,1002


Rows with NaN severity, by crop and year:
crop   year
WHEAT  2000    150
dtype: int64

Confirmed: severity >= 3 <=> at least one week at D2 or worse. So 3 = D2, 4 = D3, 5 = D4.


---
## Check 1 — How many rows per state and crop actually have a reported yield?

A "reported" row is one where `yield_per_acre` is not NaN. By construction this is the same as `yield_status == "reported"`; the cross-tab below verifies that.

In [5]:
has_yield = df[COL_YIELD].notna()

# yield_status is the master's column that records WHY a yield is missing.
xt = pd.crosstab(df[COL_YIELD_STATUS], has_yield.map({True: "yield present", False: "yield NaN"}))
display(xt)
assert (has_yield == (df[COL_YIELD_STATUS] == "reported")).all(), "yield NaN does not line up with yield_status"

overall_pct = 100 * has_yield.mean()
print(f"Overall: {has_yield.sum():,} of {len(df):,} rows have a yield ({overall_pct:.1f}%)")

yield_per_acre,yield NaN,yield present
yield_status,,
not_reported,5343,0
reported,0,11639
suppressed,74,0


Overall: 11,639 of 17,056 rows have a yield (68.2%)


In [6]:
# Coverage table: one line per state x crop
def coverage(g):
    y = g[g[COL_YIELD].notna()]
    return pd.Series({
        "rows_total": len(g),
        "rows_with_yield": len(y),
        "pct_with_yield": round(100 * len(y) / len(g), 1),
        "counties_total": g[COL_FIPS].nunique(),
        "counties_with_yield": y[COL_FIPS].nunique(),
        "years_with_yield": y[COL_YEAR].nunique(),
        "first_year_with_yield": y[COL_YEAR].min() if len(y) else np.nan,
        "last_year_with_yield": y[COL_YEAR].max() if len(y) else np.nan,
    })

cov = df.groupby([COL_STATE, COL_CROP]).apply(coverage, include_groups=False)
cov[["rows_total", "rows_with_yield", "counties_total", "counties_with_yield", "years_with_yield"]] = \
    cov[["rows_total", "rows_with_yield", "counties_total", "counties_with_yield", "years_with_yield"]].astype(int)
display(cov)

print("Totals:")
print(cov[["rows_total", "rows_with_yield"]].sum())

rows_total  rows_with_yield  pct_with_yield  counties_total  counties_with_yield  years_with_yield  first_year_with_yield  last_year_with_yield
state_alpha crop                                                                                                                                                     
CA          CORN             806              261            32.4              31                   27                23                 2000.0                2023.0
            SORGHUM          468               51            10.9              18                   15                 7                 2002.0                2008.0
            WHEAT            884              385            43.6              34                   32                25                 2000.0                2025.0
DE          CORN              78               78           100.0               3                    3                26                 2000.0                2025.0
            SORGHUM           78               10            12.8               3                    2                 5                 2000.0                2004.0
            SOYBEANS          78               72            92.3               3                    3                26                 2000.0                2025.0
            WHEAT             78               30            38.5               3                    3                10                 2000.0                2009.0
IA          CORN            2574             2470            96.0              99                   99                26                 2000.0                2025.0
            SOYBEANS        2574             2437            94.7              99                   99                26                 2000.0                2025.0
            WHEAT            728              103            14.1              28                   28                 9                 2000.0                2008.0
NE          CORN            2366             2069            87.4              91                   91                26                 2000.0                2025.0
            SORGHUM         2028              629            31.0              78                   71                26                 2000.0                2025.0
            SOYBEANS        2106             1732            82.2              81                   81                26                 2000.0                2025.0
            WHEAT           2210             1312            59.4              85                   85                25                 2000.0                2025.0

Totals:
rows_total         17056
rows_with_yield    11639
dtype: int64


In [7]:
# Missing rows broken down by reason (yield_status), by state x crop.
# Nothing is filled or dropped here; this just counts.
missing = df[df[COL_YIELD].isna()]
reason = (missing.groupby([COL_STATE, COL_CROP, COL_YIELD_STATUS]).size()
                 .unstack(COL_YIELD_STATUS, fill_value=0))
reason["missing_total"] = reason.sum(axis=1)
display(reason)
print(reason[["not_reported", "suppressed"]].sum())

yield_status          not_reported  suppressed  missing_total
state_alpha crop                                             
CA          CORN               516          29            545
            SORGHUM            401          16            417
            WHEAT              491           8            499
DE          SORGHUM             67           1             68
            SOYBEANS             6           0              6
            WHEAT               48           0             48
IA          CORN               104           0            104
            SOYBEANS           137           0            137
            WHEAT              625           0            625
NE          CORN               297           0            297
            SORGHUM           1379          20           1399
            SOYBEANS           374           0            374
            WHEAT              898           0            898

yield_status
not_reported    5343
suppressed        74
dtype: int64


In [8]:
# Does the raw NASS file carry a reason for a missing yield? It has no status column.
# What it has: 'value_raw' is the text NASS published; the master's 'suppressed' rows correspond to a
# YIELD record whose value_raw is '0' (planted, no grain harvested). 'not_reported' rows have no YIELD
# record in nass_raw at all.
nass = load_nass_raw()
print("nass_raw columns:", list(nass.columns))
y = nass[nass["statistic"] == "YIELD"]
print("\nYIELD records:", len(y))
print("  value is NaN (value_raw not numeric):", y["value"].isna().sum())
print("  value_raw of those:", y.loc[y["value"].isna(), "value_raw"].value_counts().to_dict())
print("  numeric yields:", y["value"].notna().sum())

# Reconcile with the master
nass_keys = y.set_index([COL_FIPS, COL_CROP, COL_YEAR]).index
m_idx = df.set_index([COL_FIPS, COL_CROP, COL_YEAR])
in_nass = m_idx.index.isin(nass_keys)
print("\nMaster rows with a YIELD record in nass_raw, by yield_status:")
print(pd.crosstab(pd.Series(df[COL_YIELD_STATUS].values, name="yield_status"),
                  pd.Series(in_nass, name="has_nass_YIELD_record")))

nass_raw columns: ['state', 'state_abbr', 'state_fips', 'county', 'county_fips', 'year', 'crop', 'description', 'statistic', 'unit', 'value_raw', 'value', 'fips']

YIELD records: 11713
  value is NaN (value_raw not numeric): 74
  value_raw of those: {'0': 74}
  numeric yields: 11639

Master rows with a YIELD record in nass_raw, by yield_status:
has_nass_YIELD_record  False  True 
yield_status                       
not_reported            5343      0
reported                   0  11639
suppressed                 0     74


In [9]:
# Sanity check against the known figures.
state_crop_reported = cov["rows_with_yield"]
actual = {
    "rows_total": len(df),
    "rows_with_yield": int(has_yield.sum()),
    "pct_with_yield": round(overall_pct),
    "not_reported": int((df[COL_YIELD_STATUS] == "not_reported").sum()),
    "suppressed": int((df[COL_YIELD_STATUS] == "suppressed").sum()),
    "iowa_corn_reported_approx": int(state_crop_reported.loc[("IA", "CORN")]),
    "delaware_corn_reported": int(state_crop_reported.loc[("DE", "CORN")]),
}
sanity = pd.DataFrame({"known": KNOWN, "actual": actual})
sanity["match"] = np.where(
    sanity.index.str.endswith("approx"),
    (sanity["actual"] - sanity["known"]).abs() <= 50,   # "about 2,500": allow ±50
    sanity["known"] == sanity["actual"],
)
display(sanity)
if not sanity["match"].all():
    print("MISMATCH on:", list(sanity.index[~sanity["match"]]))
else:
    print("All known figures matched.")

,known,actual,match
rows_total,17056,17056,True
rows_with_yield,11639,11639,True
pct_with_yield,68,68,True
not_reported,5343,5343,True
suppressed,74,74,True
iowa_corn_reported_approx,2500,2470,True
delaware_corn_reported,78,78,True


All known figures matched.


### Iowa corn and Iowa soybeans: how many of the 26 years does each county report?

In [10]:
ia = df[(df[COL_STATE] == FOCUS_STATE) & (df[COL_CROP].isin(FOCUS_CROPS)) & df[COL_YIELD].notna()]

years_per_county = (ia.groupby([COL_CROP, COL_FIPS])[COL_YEAR].nunique()
                      .rename("years_reported"))

# Distribution: how many counties report N of the 26 years
dist = (years_per_county.groupby(COL_CROP).value_counts()
                        .unstack(COL_CROP, fill_value=0)
                        .sort_index(ascending=False))
dist.index.name = "years_reported (of 26)"
dist.columns = [f"{c}: n_counties" for c in dist.columns]
display(dist)

print("Summary of years reported per county:")
display(years_per_county.groupby(COL_CROP).describe()[["count", "mean", "min", "25%", "50%", "75%", "max"]])

for crop in FOCUS_CROPS:
    s = years_per_county.loc[crop]
    print(f"{crop}: {len(s)} counties with any yield; "
          f"{(s == N_YEARS_EXPECTED).sum()} report all 26 years; "
          f"{(s >= 20).sum()} report 20+ years; "
          f"{(s < 10).sum()} report fewer than 10 years")

,CORN: n_counties,SOYBEANS: n_counties
years_reported (of 26),,
26,43,35
25,30,28
24,12,14
23,10,12
22,2,7
21,0,2
20,2,0
19,0,1


Summary of years reported per county:


,count,mean,min,25%,50%,75%,max
crop,,,,,,,
CORN,99.0,24.949495,20.0,24.0,25.0,26.0,26.0
SOYBEANS,99.0,24.616162,19.0,24.0,25.0,26.0,26.0


CORN: 99 counties with any yield; 43 report all 26 years; 99 report 20+ years; 0 report fewer than 10 years
SOYBEANS: 99 counties with any yield; 35 report all 26 years; 98 report 20+ years; 0 report fewer than 10 years


In [11]:
# Same thing the other way round: how many Iowa counties report in each year, per crop.
by_year = (ia.groupby([COL_YEAR, COL_CROP])[COL_FIPS].nunique()
             .unstack(COL_CROP, fill_value=0))
by_year.columns = [f"{c}: counties reporting" for c in by_year.columns]
display(by_year.T)

year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
CORN: counties reporting,99,99,99,99,99,99,99,99,99,99,99,99,99,99,99,95,99,95,93,88,95,84,97,87,73,79
SOYBEANS: counties reporting,99,99,99,99,99,99,99,99,99,99,99,99,99,99,99,97,99,96,96,82,86,76,97,82,68,73


In [12]:
# Which Iowa counties are the least complete? (useful when deciding whether to require full series)
least = years_per_county.reset_index().sort_values(["crop", "years_reported"]).groupby(COL_CROP).head(10)
least = least.merge(df[[COL_FIPS, "county_name"]].drop_duplicates(), on=COL_FIPS, how="left")
display(least)

,crop,fips,years_reported,county_name
0,CORN,19003,20,Adams County
1,CORN,19039,20,Clarke County
2,CORN,19117,22,Lucas County
3,CORN,19159,22,Ringgold County
4,CORN,19007,23,Appanoose County
5,CORN,19053,23,Decatur County
6,CORN,19063,23,Emmet County
7,CORN,19077,23,Guthrie County
8,CORN,19085,23,Harrison County
9,CORN,19099,23,Jasper County


---
## Check 2 — How many county-years are in drought at each severity level?

Counts below use **only rows with a reported yield** (`yield_per_acre` not NaN), because those are the rows any yield-loss analysis can use. One row = one county, one crop, one year.

`D2+ total` = severity ≥ 3 (D2, D3, D4). `D3+ total` = severity ≥ 4 (D3, D4).

In [13]:
rep = df[df[COL_YIELD].notna()].copy()
rep["sev_label"] = rep[COL_SEV].map(SEV_LABELS)

sev_tab = (rep.groupby([COL_STATE, COL_CROP])[COL_SEV]
              .value_counts(dropna=False)
              .unstack(COL_SEV, fill_value=0))
# make sure all 0..5 columns exist even if a group lacks one, and keep NaN as its own column
for v in range(6):
    if v not in sev_tab.columns:
        sev_tab[v] = 0
nan_col = [c for c in sev_tab.columns if pd.isna(c)]
sev_tab["sev_NaN"] = sev_tab[nan_col].sum(axis=1) if nan_col else 0
sev_tab = sev_tab.drop(columns=nan_col)
sev_tab = sev_tab[[0, 1, 2, 3, 4, 5, "sev_NaN"]].astype(int)
sev_tab["D2+ total"] = sev_tab[[SEV_D2, SEV_D3, SEV_D4]].sum(axis=1)
sev_tab["D3+ total"] = sev_tab[[SEV_D3, SEV_D4]].sum(axis=1)
sev_tab["rows_with_yield"] = rep.groupby([COL_STATE, COL_CROP]).size()
sev_tab["pct_D2+"] = (100 * sev_tab["D2+ total"] / sev_tab["rows_with_yield"]).round(1)
sev_tab["pct_D3+"] = (100 * sev_tab["D3+ total"] / sev_tab["rows_with_yield"]).round(1)
sev_tab = sev_tab.rename(columns={v: f"{v}={SEV_LABELS[v]}" for v in range(6)})
display(sev_tab)

print("Reported rows with NaN severity (should be WHEAT 2000 only):")
print(rep[rep[COL_SEV].isna()].groupby([COL_STATE, COL_CROP, COL_YEAR]).size())

max_drought_severity  0=none  1=D0  2=D1  3=D2  4=D3  5=D4  sev_NaN  D2+ total  D3+ total  rows_with_yield  pct_D2+  pct_D3+
state_alpha crop                                                                                                            
CA          CORN         101    37    43    44    17    19        0         80         36              261     30.7     13.8
            SORGHUM       16    14    15     6     0     0        0          6          0               51     11.8      0.0
            WHEAT         12    89    80    99    38    35       32        172         73              385     44.7     19.0
DE          CORN          14    33    21     7     0     3        0         10          3               78     12.8      3.8
            SORGHUM        6     0     2     0     0     2        0          2          2               10     20.0     20.0
            SOYBEANS      12    29    18    10     0     3        0         13          3               72     18.1      4.2
            WHEAT          5     4    10     3     5     0        3          8          5               30     26.7     16.7
IA          CORN         430   714   564   485   261    16        0        762        277             2470     30.9     11.2
            SOYBEANS     377   640   584   519   297    20        0        836        317             2437     34.3     13.0
            WHEAT          6    28    19    28    16     0        6         44         16              103     42.7     15.5
NE          CORN         207   544   432   409   321   156        0        886        477             2069     42.8     23.1
            SORGHUM       36   136   136   142   139    40        0        321        179              629     51.0     28.5
            SOYBEANS     139   447   367   371   282   126        0        779        408             1732     45.0     23.6
            WHEAT        106   264   286   259   261    72       64        592        333             1312     45.1     25.4

Reported rows with NaN severity (should be WHEAT 2000 only):
state_alpha  crop   year
CA           WHEAT  2000    32
DE           WHEAT  2000     3
IA           WHEAT  2000     6
NE           WHEAT  2000    64
dtype: int64


### Iowa, by crop: how many distinct counties and years have D2+ and D3+?

In [14]:
ia_rep = rep[(rep[COL_STATE] == FOCUS_STATE) & (rep[COL_CROP].isin(FOCUS_CROPS))]

rows = []
for crop, g in ia_rep.groupby(COL_CROP):
    d2 = g[g[COL_SEV] >= SEV_D2]
    d3 = g[g[COL_SEV] >= SEV_D3]
    rows.append({
        "crop": crop,
        "rows_with_yield": len(g),
        "D2+ county-years": len(d2),
        "D2+ distinct counties": d2[COL_FIPS].nunique(),
        "D2+ distinct years": d2[COL_YEAR].nunique(),
        "D2+ years": ", ".join(str(y) for y in sorted(d2[COL_YEAR].unique())),
        "D3+ county-years": len(d3),
        "D3+ distinct counties": d3[COL_FIPS].nunique(),
        "D3+ distinct years": d3[COL_YEAR].nunique(),
        "D3+ years": ", ".join(str(y) for y in sorted(d3[COL_YEAR].unique())),
    })
ia_summary = pd.DataFrame(rows).set_index("crop")
with pd.option_context("display.max_colwidth", None):
    display(ia_summary.T)

crop,CORN,SOYBEANS
rows_with_yield,2470,2437
D2+ county-years,762,836
D2+ distinct counties,99,99
D2+ distinct years,17,17
D2+ years,"2000, 2002, 2003, 2005, 2006, 2007, 2011, 2012, 2013, 2014, 2017, 2018, 2020, 2021, 2022, 2023, 2024","2000, 2002, 2003, 2005, 2006, 2007, 2011, 2012, 2013, 2014, 2017, 2018, 2020, 2021, 2022, 2023, 2024"
D3+ county-years,277,317
D3+ distinct counties,99,99
D3+ distinct years,11,11
D3+ years,"2000, 2003, 2005, 2012, 2013, 2017, 2018, 2020, 2021, 2022, 2023","2000, 2003, 2005, 2012, 2013, 2017, 2018, 2020, 2021, 2022, 2023"


In [15]:
# Year-by-crop table: number of Iowa counties (with a reported yield) at D3+ and at D2+ each year.
def counties_at(g, thresh):
    return g.loc[g[COL_SEV] >= thresh, COL_FIPS].nunique()

yr = ia_rep.groupby([COL_YEAR, COL_CROP])
year_tab = pd.DataFrame({
    "CORN: counties D3+":     yr.apply(lambda g: counties_at(g, SEV_D3), include_groups=False).unstack(COL_CROP)["CORN"],
    "SOYBEANS: counties D3+": yr.apply(lambda g: counties_at(g, SEV_D3), include_groups=False).unstack(COL_CROP)["SOYBEANS"],
    "CORN: counties D2+":     yr.apply(lambda g: counties_at(g, SEV_D2), include_groups=False).unstack(COL_CROP)["CORN"],
    "SOYBEANS: counties D2+": yr.apply(lambda g: counties_at(g, SEV_D2), include_groups=False).unstack(COL_CROP)["SOYBEANS"],
    "CORN: counties reporting":     yr.size().unstack(COL_CROP)["CORN"],
    "SOYBEANS: counties reporting": yr.size().unstack(COL_CROP)["SOYBEANS"],
}).fillna(0).astype(int)
display(year_tab)

# How concentrated is D3+ in a single year?
for crop in FOCUS_CROPS:
    col = f"{crop}: counties D3+"
    tot = year_tab[col].sum()
    top = year_tab[col].sort_values(ascending=False).head(5)
    print(f"{crop}: {tot} D3+ county-years in total. Top years: "
          + ", ".join(f"{y} ({n}, {100*n/tot:.0f}%)" for y, n in top.items() if n > 0))

,CORN: counties D3+,SOYBEANS: counties D3+,CORN: counties D2+,SOYBEANS: counties D2+,CORN: counties reporting,SOYBEANS: counties reporting
year,,,,,,
2000,14,26,83,83,99,99
2001,0,0,0,0,99,99
2002,0,0,14,14,99,99
2003,31,48,92,92,99,99
2004,0,0,0,0,99,99
2005,15,15,26,47,99,99
2006,0,0,51,51,99,99
2007,0,0,10,10,99,99
2008,0,0,0,0,99,99


CORN: 277 D3+ county-years in total. Top years: 2012 (92, 33%), 2023 (41, 15%), 2003 (31, 11%), 2021 (25, 9%), 2022 (21, 8%)
SOYBEANS: 317 D3+ county-years in total. Top years: 2012 (93, 29%), 2003 (48, 15%), 2023 (44, 14%), 2000 (26, 8%), 2020 (24, 8%)


In [16]:
# Severity breakdown by year for Iowa (all levels), so you can see what the non-D3 years look like.
sev_by_year = (ia_rep.groupby([COL_CROP, COL_YEAR])[COL_SEV].value_counts()
                     .unstack(COL_SEV, fill_value=0))
sev_by_year.columns = [f"{int(v)}={SEV_LABELS[int(v)]}" for v in sev_by_year.columns]
display(sev_by_year)

0=none  1=D0  2=D1  3=D2  4=D3  5=D4
crop     year                                      
CORN     2000       0     0    16    69    14     0
         2001       0    82    17     0     0     0
         2002      22    24    39    14     0     0
         2003       0     0     7    61    31     0
         2004      22    74     3     0     0     0
         2005      20    16    37    11    15     0
         2006       4    14    30    51     0     0
         2007      19    36    34    10     0     0
         2008      55    30    14     0     0     0
         2009      54    42     3     0     0     0
         2010      99     0     0     0     0     0
         2011       4    56    23    16     0     0
         2012       0     0     0     7    87     5
         2013       0     6    29    58     6     0
         2014       4    39    49     7     0     0
         2015      56    33     6     0     0     0
         2016      16    53    30     0     0     0
         2017       1    19    49    17     9     0
         2018      35    31    13    11     3     0
         2019       6    50    32     0     0     0
         2020       0     4    42    29    20     0
         2021       0     7    20    32    25     0
         2022       4    23    26    23    20     1
         2023       0     0     1    45    31    10
         2024       0    26    23    24     0     0
         2025       9    49    21     0     0     0
SOYBEANS 2000       0     0    16    57    26     0
         2001       0    82    17     0     0     0
         2002      22    24    39    14     0     0
         2003       0     0     7    44    48     0
         2004       0    96     3     0     0     0
         2005      11    14    27    32    15     0
         2006       4    14    30    51     0     0
         2007      19    36    34    10     0     0
         2008      55    29    15     0     0     0
         2009      54    42     3     0     0     0
         2010      99     0     0     0     0     0
         2011       2    23    41    33     0     0
         2012       0     0     0     6    88     5
         2013       0     6    28    59     6     0
         2014       4    39    49     7     0     0
         2015      47    34    16     0     0     0
         2016      16    53    30     0     0     0
         2017       1    18    47    20    10     0
         2018      35    31    13    11     6     0
         2019       4    48    30     0     0     0
         2020       0     4    38    20    24     0
         2021       0     6    17    30    23     0
         2022       0     3    33    39    21     1
         2023       0     0     1    37    30    14
         2024       0     1    18    49     0     0
         2025       4    37    32     0     0     0

---
## Notes for the write-up

- `yield_status` in the master is the column that records why a yield is missing (`not_reported` vs `suppressed`). The raw NASS file has no such column; `suppressed` corresponds to a NASS `YIELD` record published as `0`, and `not_reported` to no NASS yield record at all.
- The master has **no irrigation column**. The data dictionary says Iowa is rainfed and that NASS publishes no irrigated series for Iowa, so the irrigation extension has no Iowa rows.
- Severity NaN only occurs for winter wheat in 2000 (USDM archive starts January 2000); it does not affect Iowa corn or soybeans.